In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from tqdm import tqdm
import os
import contextlib

In [ ]:
df = pd.read_csv("../input/datasetsef/trainDailyM4.csv",dtype=str)
dft = pd.read_csv("../input/datasetsef/testDailyM4.csv", low_memory=False)

index = list(df.columns)

dft=dft.loc[3:,:]

datax=pd.concat([df, dft], axis=0, ignore_index=True)

In [ ]:
def train_test_split(data, n_test):
    return data[:-n_test, :], data[-n_test:, :]

def calculate_zeros_percentage(arr):
    num_zeros = np.count_nonzero(arr == 0)
    total_elements = arr.size
    zeros_percentage = (num_zeros / total_elements) * 100
    return zeros_percentage
    
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    n_vars = 1 if type(data) is list else data.shape[1]
    df = pd.DataFrame(data)
    cols = list()
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
    for i in range(0, n_out):
        cols.append(df.shift(-i))
    agg = pd.concat(cols, axis=1)
    if dropnan:
        agg.dropna(inplace=True)
    return agg.values


def prediction_plot(testY, test_predict):

    len_prediction=[x for x in range(len(testY))]
    plt.figure(figsize=(8,4))
    plt.plot(len_prediction, testY[:len(testY)], marker='.', label="actual")
    plt.plot(len_prediction, test_predict[:len(testY)], 'r', label="prediction")
    plt.tight_layout()
    sns.despine(top=True)
    plt.subplots_adjust(left=0.07)
    plt.ylabel('Pred Trend', size=15)
    plt.xlabel('Time step', size=15)
    plt.legend(fontsize=15)
    plt.show();
    
params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.09,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.75,
    "bagging_freq": 1,
    "lambda_l2": 0.1,
    "verbosity": -1,
    "num_iterations": 2000,
    "num_leaves": 32,
    "min_data_in_leaf": 50,
}


In [ ]:

callbacks = [lgb.early_stopping(200), lgb.log_evaluation(0)]

smapeList = []
predsList = []
num_series = len(index)

for i in tqdm(range(num_series), desc="Entrenando y prediciendo", unit="serie"):
    nMA = datax.loc[3:, index[i]]
    nMA = nMA.dropna()
    values = nMA.values.reshape(-1, 1)
    
    data = series_to_supervised(values, n_in=6)
    
    train_val, test = train_test_split(data, 14)  # 14-step forecast horizon
    val_size = int(len(train_val) * 0.2)
    train, val = train_val[:-val_size], train_val[-val_size:]
    
    trainX, trainY = train[:, :-1], train[:, -1]
    valX, valY = val[:, :-1], val[:, -1]
    testX, testY = test[:, :-1], test[:, -1]
    
    train_data = lgb.Dataset(trainX, label=trainY)
    valid_data = lgb.Dataset(valX, label=valY)
    
    m_lgb = lgb.train(
        params,
        train_data,
        valid_sets=[train_data, valid_data],
        num_boost_round=3600,
        callbacks=callbacks
    )
    
    preds = m_lgb.predict(testX)
    predsList.append(preds)

In [ ]:

dfp=pd.DataFrame(predsList, index=index)
dfp=dfp.transpose()
dfp.to_csv("preds_LGBM_M4Dailyv_A5M5.csv")